# Identity Router Real-World Demo (Colab Ready)


This notebook runs a practical demo of the Identity Router components on a real image.


What this notebook covers:


- Install dependencies in Colab


- Load project code from this repository


- Extract ArcFace embedding from a real reference photo


- Build localized identity mask (fallback + pose-guided)


- Visualize outputs step by step




Notes:


- Use a clear frontal face photo for best embedding quality


- DWPose and SAM are optional in this notebook; we demonstrate pose-guided masking directly with keypoints


- You can extend this to real DWPose/SAM models later

In [ ]:
# Step 1: Install dependencies (Colab)

%pip -q install insightface onnxruntime opencv-python numpy matplotlib



import os

import sys

from pathlib import Path



import cv2

import numpy as np

import matplotlib.pyplot as plt



print("[OK] Dependencies installed and imports loaded")

In [ ]:

from pathlib import Path



import cv2

import numpy as np

import matplotlib.pyplot as plt



print("[OK] Dependencies installed and imports loaded")

In [ ]:
# Step 2: Point to project root and import Identity Router functions

# If this notebook runs inside the repo, this should already work.

# In Colab, set PROJECT_ROOT to your cloned/mounted repo path.



PROJECT_ROOT = Path('/content/pivot')

if PROJECT_ROOT.exists():

    sys.path.insert(0, str(PROJECT_ROOT))

    print(f"[INFO] Using project root: {PROJECT_ROOT}")

else:

    print("[WARN] /content/pivot not found. Update PROJECT_ROOT to your repo path before running imports.")



from core.identity_router import extract_arcface_embedding, build_localized_identity_mask

print("[OK] Imported extract_arcface_embedding and build_localized_identity_mask")

In [ ]:
# Step 3: Upload a real reference image
from google.colab import files

uploaded = files.upload()
assert uploaded, "Please upload at least one image file"

reference_path = next(iter(uploaded.keys()))
print(f"[OK] Uploaded reference image: {reference_path}")

img_bgr = cv2.imread(reference_path)
assert img_bgr is not None, "OpenCV could not read the uploaded image"
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title("Uploaded Reference Image")
plt.axis('off')
plt.show()

In [ ]:
# Step 4: Extract ArcFace embedding and inspect it
embedding = extract_arcface_embedding(reference_path, model_name='buffalo_l', ctx_id=0)

print('[OK] Embedding extracted')
print('Shape:', embedding.shape)
print('Dtype:', embedding.dtype)
print('L2 norm:', float(np.linalg.norm(embedding)))
print('First 8 values:', np.round(embedding[:8], 4))

In [ ]:
# Step 5: Build fallback localized mask (face-driven)
mask_fallback = build_localized_identity_mask(reference_path)

print('[OK] Fallback mask generated')
print('Mask shape:', mask_fallback.shape)
print('Mask min/max:', float(mask_fallback.min()), float(mask_fallback.max()))
print('Non-zero pixels:', int((mask_fallback > 0).sum()))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title('Reference Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(mask_fallback, cmap='gray')
plt.title('Fallback Mask')
plt.axis('off')

plt.subplot(1, 3, 3)
overlay = img_rgb.copy()
overlay[mask_fallback <= 0.1] = (overlay[mask_fallback <= 0.1] * 0.25).astype(np.uint8)
plt.imshow(overlay)
plt.title('Masked Region Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Build pose-guided localized mask (DWPose-like keypoints demo)
# We build synthetic keypoints anchored near the image center to demonstrate the pose path.
# In production, feed real DWPose keypoints into pose_keypoints.

h, w = img_bgr.shape[:2]
cx, cy = w // 2, h // 2

pose_keypoints = np.array([
    [cx, cy - 45, 0.95],
    [cx - 8, cy - 50, 0.90],
    [cx + 8, cy - 50, 0.90],
    [cx - 14, cy - 40, 0.90],
    [cx + 14, cy - 40, 0.90],
    [cx - 12, cy - 20, 0.95],
    [cx + 12, cy - 20, 0.95],
    [cx - 20, cy + 0, 0.90],
    [cx + 20, cy + 0, 0.90],
    [cx - 24, cy + 20, 0.85],
    [cx + 24, cy + 20, 0.85],
    [cx - 10, cy + 20, 0.95],
    [cx + 10, cy + 20, 0.95],
    [cx - 12, cy + 45, 0.90],
    [cx + 12, cy + 45, 0.90],
    [cx - 14, cy + 65, 0.85],
    [cx + 14, cy + 65, 0.85],
], dtype=np.float32)

mask_pose = build_localized_identity_mask(reference_path, pose_keypoints=pose_keypoints)

print('[OK] Pose-guided mask generated')
print('Mask shape:', mask_pose.shape)
print('Mask min/max:', float(mask_pose.min()), float(mask_pose.max()))
print('Non-zero pixels:', int((mask_pose > 0).sum()))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title('Reference Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(mask_pose, cmap='gray')
plt.title('Pose-Guided Mask')
plt.axis('off')

plt.subplot(1, 3, 3)
overlay_pose = img_rgb.copy()
overlay_pose[mask_pose <= 0.1] = (overlay_pose[mask_pose <= 0.1] * 0.25).astype(np.uint8)
plt.imshow(overlay_pose)
plt.title('Pose Mask Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Optional SAM-style refinement demo with a mock predictor
class MockSAMPredictor:
    def set_image(self, image):
        self.image = image

    def predict(self, box=None, multimask_output=False):
        m = np.zeros((h, w), dtype=np.float32)
        x1, y1, x2, y2 = map(int, box)
        m[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = 1.0
        return (np.expand_dims(m, 0), None, None)

mask_sam_demo = build_localized_identity_mask(
    reference_path,
    pose_keypoints=pose_keypoints,
    sam_predictor=MockSAMPredictor(),
)

print('[OK] SAM-style refinement path executed (mock predictor)')
print('Mask shape:', mask_sam_demo.shape)
print('Mask min/max:', float(mask_sam_demo.min()), float(mask_sam_demo.max()))
print('Non-zero pixels:', int((mask_sam_demo > 0).sum()))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(mask_pose, cmap='gray')
plt.title('Pose Mask')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(mask_sam_demo, cmap='gray')
plt.title('Pose + SAM-Style Mask')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(np.clip(mask_sam_demo - mask_pose, 0, 1), cmap='magma')
plt.title('Refinement Difference')
plt.axis('off')

plt.tight_layout()
plt.show()